In [10]:
import cv2
import numpy as np
from numpy.fft import fft2, ifft2
import os
import glob

# --- Helper Functions (same as before) ---

def create_motion_blur_kernel(size, angle):
    """Creates a motion blur kernel."""
    kernel = np.zeros((size, size))
    center = size // 2
    cv2.line(kernel, (0, center), (size - 1, center), 1.0, 1)
    M = cv2.getRotationMatrix2D((center, center), angle, 1)
    motion_kernel = cv2.warpAffine(kernel, M, (size, size))
    motion_kernel = motion_kernel / motion_kernel.sum()
    return motion_kernel

def apply_degradation(image, kernel, noise_std_dev):
    """Applies a blur kernel and adds Gaussian noise to an image."""
    image_fft = fft2(image)
    kernel_fft = fft2(kernel, s=image.shape)
    blurred_fft = image_fft * kernel_fft
    blurred_image = np.real(ifft2(blurred_fft))
    
    noise = np.random.normal(0, noise_std_dev, image.shape)
    noisy_image = blurred_image + noise
    
    degraded_image = np.clip(noisy_image, 0, 255)
    return degraded_image

def inverse_filter(degraded_image, kernel):
    """Applies inverse filtering to restore an image."""
    degraded_fft = fft2(degraded_image)
    kernel_fft = fft2(kernel, s=degraded_image.shape)
    
    epsilon = 1e-8
    restored_fft = degraded_fft / (kernel_fft + epsilon)
    
    restored_image = np.real(ifft2(restored_fft))
    return np.clip(restored_image, 0, 255)

def wiener_filter(degraded_image, kernel, K):
    """Applies Wiener filtering to restore an image."""
    degraded_fft = fft2(degraded_image)
    kernel_fft = fft2(kernel, s=degraded_image.shape)
    
    kernel_fft_conj = np.conj(kernel_fft)
    kernel_fft_mag_sq = np.abs(kernel_fft)**2
    
    wiener_fft = (kernel_fft_conj / (kernel_fft_mag_sq + K)) * degraded_fft
    
    restored_image = np.real(ifft2(wiener_fft))
    return np.clip(restored_image, 0, 255)

# --- Main Batch Processing Logic ---

def process_image_folder():
    """Processes all images in the 'images' folder and saves results."""
    # Define input and output directories
    input_folder = 'images'
    output_folder = 'output'
    
    # Create output directory if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"Created directory: {output_folder}")

    # Find all image files in the input folder
    image_paths = glob.glob(os.path.join(input_folder, '*.[pP][nN][gG]')) + \
                  glob.glob(os.path.join(input_folder, '*.[jJ][pP][gG]')) + \
                  glob.glob(os.path.join(input_folder, '*.[jJ][pP][eE][gG]'))

    if not image_paths:
        print(f"No images found in the '{input_folder}' directory. Please add some images.")
        return

    print(f"Found {len(image_paths)} images to process.")

    # Define degradation parameters
    blur_size = 21
    blur_angle = 45
    noise_std_dev = 10
    K_wiener = 0.01  # K constant for Wiener filter

    # Create the degradation kernel (motion blur)
    motion_kernel = create_motion_blur_kernel(blur_size, blur_angle)

    # Loop through each image
    for i, image_path in enumerate(image_paths):
        # Get the base filename
        base_name = os.path.basename(image_path)
        name, ext = os.path.splitext(base_name)
        print(f"Processing ({i+1}/{len(image_paths)}): {base_name}")

        # Load the image in grayscale
        original_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if original_image is None:
            print(f"  - Could not read image, skipping.")
            continue
        original_image = original_image.astype(float)

        # Apply degradation (blur + noise)
        degraded_image = apply_degradation(original_image, motion_kernel, noise_std_dev)

        # Apply Inverse Filtering
        restored_inverse = inverse_filter(degraded_image, motion_kernel)

        # Apply Wiener Filtering
        restored_wiener = wiener_filter(degraded_image, motion_kernel, K_wiener)

        # Save the results
        cv2.imwrite(os.path.join(output_folder, f"{name}_01_original{ext}"), original_image.astype(np.uint8))
        cv2.imwrite(os.path.join(output_folder, f"{name}_02_degraded{ext}"), degraded_image.astype(np.uint8))
        cv2.imwrite(os.path.join(output_folder, f"{name}_03_restored_inverse{ext}"), restored_inverse.astype(np.uint8))
        cv2.imwrite(os.path.join(output_folder, f"{name}_04_restored_wiener{ext}"), restored_wiener.astype(np.uint8))

    print("\nProcessing complete! Check the 'output' folder for results. 📂")

# Run the main function
if __name__ == '__main__':
    process_image_folder()

Found 1 images to process.
Processing (1/1): img.jpg

Processing complete! Check the 'output' folder for results. 📂
